# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}\n")
print(f"Variables with personal sensitive information: {metadata.personalSensitiveInformation}")

## 2. Data Overview
Review available **record sets** and their constituent **fields**. All references are by their Croissant `@id`.

In [ ]:
# List all available record sets in the dataset by their @id

print("Available record sets:")
for record_set in dataset.metadata.recordSet:
    print(f"- @id: {record_set['@id']}")
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"     - @id: {field_id}")
    if 'column' in record_set:
        print("  Columns:")
        for column in record_set['column']:
            column_id = column['@id'] if isinstance(column, dict) and '@id' in column else str(column)
            print(f"     - @id: {column_id}")
print()

# For demonstration, print a record from the first record set by @id
if len(dataset.metadata.recordSet) > 0:
    example_record_set_id = dataset.metadata.recordSet[0]['@id']
    print(f"Showing a sample record from record set @id '{example_record_set_id}':")
    try:
        for x in dataset.records(record_set=example_record_set_id):
            print(x)
            break
    except Exception as e:
        print(f"Failed to fetch records from record set {example_record_set_id}: {e}")
else:
    print("No record sets defined in this dataset.")

## 3. Data Extraction
Load data from each available record set into pandas DataFrames. All references use Croissant `@id`.

> **Note:** For this dataset, there may be a principal record set containing the main tabular data. All steps reference record set and field/column IDs.

In [ ]:
# Build a list of all record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.recordSet]
print(f"Record set @ids: {record_sets}\n")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records. Fields:", list(df.columns))
        else:
            print("  No records were loaded.")
    except Exception as e:
        print(f"  Error loading records: {e}")
print()
# For display: pick the first (principal) record set if any
if len(record_sets) > 0 and record_sets[0] in dataframes:
    main_record_set_id = record_sets[0]
    print(f"Top rows of DataFrame for record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set DataFrame available for display.")

## 4. Exploratory Data Analysis (EDA)
Let us process data from the main record set, focusing on a numeric field and a group field for demonstration. **All fields are referenced by their `@id`.**

Possible numeric fields (to be determined by examining DataFrame columns): e.g. 'age_at_second_crc' or similar.

Let's look for possible numeric and group fields in the loaded DataFrame.

In [ ]:
# Identify candidate numeric and group fields by inspecting the DataFrame
from IPython.display import display

if len(dataframes) > 0:
    # Use main record set (if available)
    main_record_set_id = record_sets[0]
    df = dataframes[main_record_set_id]

    print(f"Column names for main record set '{main_record_set_id}':")
    print(list(df.columns))

    # Try to auto-select a numeric field by dtype (float or int)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to parse columns that look like age, interval, or years
        for col in df.columns:
            if any(word in col.lower() for word in ['age', 'interval', 'years', 'duration']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notnull().any():
                        numeric_fields.append(col)
                except Exception:
                    continue

    print(f"\nNumeric fields detected: {numeric_fields}")
    # Pick the first numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = None
        print("No obvious numeric field found.")

    # Try to pick a group field: look for a 'sex', 'gender', 'site', 'location' or 'msi' field
    possible_group_fields = [col for col in df.columns if any(term in col.lower() for term in ['sex', 'gender', 'site', 'location', 'msi'])]
    print(f"Possible group fields: {possible_group_fields}")
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    
    # Proceed with EDA if a numeric field is available
    if numeric_field_id:
        # Example: filter for values above threshold (e.g. age > 50)
        threshold = 50
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' values (z-score normalization):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optionally group by the group_field_id (if available)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df)
    else:
        print("No numeric field available for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship with the group field (if available).

> The visualization below automatically adapts to the field names detected in the previous step.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if there is a proper numeric field
if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (> threshold)")
    plt.show()

    # If a group field is also available, plot a boxplot
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No filtered data or numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to use the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library to load, explore, and process a FAIR-structured clinical oncology dataset defined by a Croissant schema. We:
- Dynamically loaded available record sets and their field structure by `@id`
- Extracted main record(s) and summarised key fields
- Applied filtering, normalization, and grouping/aggregation
- Visualized distributions and group-wise differences

This approach can be adapted to any Croissant-compliant dataset by referencing **all entities via their `@id`** fields.